# MNIST Training -- Unconditional, Conditional, and WGAN-GP Baseline

Trains the primal-dual OT-GAN on MNIST in three configurations, corresponding
to Sections 5.2-5.5 of the thesis:

- **Unconditional** (`train_mnist`) -- Models A/B, the anchoring and optimizer ablations.
- **Class-conditional** (`train_mnist_cond`) -- Model C, with a projection
  critic and per-class collapse detection.
- **Conditional WGAN-GP baseline** (`train_mnist_cond_wgan_gp`) -- Model CW,
  used to isolate how much of the conditional model's performance comes from
  the OT objective vs. the conditioning architecture.

Most of the cells below are historical run configs, kept commented out as a
record that we tried. The uncommented cells are the ones that produced the checkpoints referenced in the thesis tables.

Reusable code lives in `src/`: `models.py` (network architectures),
`optimizers.py` (update rules), `data.py` (`get_mnist_loaders`),
`metrics.py` (`grad_norm_l2`, `compute_w2_squared`), `plotting.py`
(`plot_generated_images`, `plot_conditional_grid`), and `training.py`
(`train_mnist`, `train_mnist_cond`, `train_mnist_cond_wgan_gp`, and their
shared TensorBoard logging helpers).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))  # make src/ importable

import torch
import matplotlib.pyplot as plt

from data import get_mnist_loaders
from models import MNISTGenerator, MNISTCritic, CondGenerator, ProjCritic
from optimizers import adam_update, optimistic_adam_update, anchored_adam_update, anchored_optimistic_adam_update
from training import train_mnist, train_mnist_cond, train_mnist_cond_wgan_gp, cond_gradient_penalty

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.set_num_threads(os.cpu_count())

my_seed = 42
torch.manual_seed(my_seed)
torch.cuda.manual_seed_all(my_seed)


def _count(m):
    """Quick param-count sanity check, used throughout the experiment cells below."""
    return sum(p.numel() for p in m.parameters())

/home/sikha/bse/NOT-GAN/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


---

## 2. Experiments



## Model A: Anchored Adam
Shared: lr=1e-4, σ=0.01, 30k MNIST, 200 epochs.

#### **A1 / gStep_fD64**
γ: stepped 1e-5 → 1e-6 @ ep100. fG=64, fD=64. kc=1. reset=50.

In [ ]:
#
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=64).to(device)
# D = MNISTCritic(channels_img=1, features_d=64, img_size=IMG_SIZE).to(device)

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = (1e-5, 1e-6, 100),
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 1,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A1/gStep_fD64',
#     reset_every   = 50,
#     device        = device,
# )

#### **A2 / gStep_fD128**
γ: stepped 1e-5 → 1e-6 @ ep100. fG=64, fD=128. kc=1. reset=50.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=64).to(device)
# D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

# # Param-count sanity check — D should be ~4× E15's D weights.
# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}  (E15 baseline ≈ 2.76M for features_d=64)')

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = (1e-5, 1e-6, 100),   # same stepped schedule as E15
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 1,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A2/gStep_fD128',
#     reset_every   = 50,
#     device        = device,
# )


#### **A3 / g1e6_kc2**:γ=1e-6. fG=128, fD=128. kc=2. reset=50.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=128).to(device)
# D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 1e-6,
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A3/g1e6_kc2',
#     device        = device,
#     reset_every   = 50,
#     ema_decay     = None,
# )




#### **A4 / g1e5_r10**
γ=1e-5. fG=128, fD=128. kc=1. reset=10.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=128).to(device)
# D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 1e-5,
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 1,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A4/g1e5_r10',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )





#### **A5 / g1e5_r10_kc2**
γ=1e-5. fG=128, fD=128. kc=2. reset=10.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=128).to(device)
# D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 1e-5,
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A5/g1e5_r10_kc2',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )





#### **A6 / g3e6_kc2**
γ=3e-6. fG=128, fD=128. kc=2. reset=10.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=128).to(device)
# D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = 0.01,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'A6/g3e6_kc2',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )

#### **A7 / g1e5_r10_ema99**
A4 + EMA of .99 only
γ=1e-5. fG=128, fD=128. kc=1. reset=10 ema=99

In [ ]:
IMG_SIZE   = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=128).to(device)
D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

def _count(m): return sum(p.numel() for p in m.parameters())
print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist(
    G, D, anchored_adam_update, train_loader,
    resume_G_path=None,
    resume_D_path=None,
    initial_noise_type = "low_freq",
    gamma         = 1e-5,
    eta           = 1e-4,
    sigma         = 0.01,
    beta1         = b1,
    anchor_power  = 1,
    k_critic      = 1,
    k_generator   = 1,
    warmup_steps  = 30_000/256 * 30,
    warmup_k_critic = 5,
    num_epochs    = 200,
    seed          = my_seed,
    run_name      = 'A7/g1e5_r10_ema99',
    device        = device,
    reset_every   = 10,
    ema_decay     = .99,
)

G params: 49,813,504
D params: 11,016,321
Steps/epoch: 2  |  Total steps: 6
EMA enabled: decay=0.99  (effective window ≈ 99 G-steps)


Epochs:   0%|          | 0/3 [00:00<?, ?it/s]/home/sikha/bse/NOT-GAN/.venv/lib/python3.13/site-packages/ot/datasets.py:204: SyntaxWarning: invalid escape sequence '\d'
  positive floating numbers corresponding to the isotropic variances in the principal subspace, for the source and target distributions, respectively. The same as \delta in :ref:`[1] <references-make_gauss-hd>`, Proposition 2.2
Epochs: 100%|██████████| 3/3 [01:54<00:00, 38.17s/it]


In [ ]:
IMG_SIZE   = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=128).to(device)
D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

def _count(m): return sum(p.numel() for p in m.parameters())
print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist(
    G, D, anchored_adam_update, train_loader,
    resume_G_path=None,
    resume_D_path=None,
    initial_noise_type = "low_freq",
    gamma         = 1e-5,
    eta           = 1e-4,
    sigma         = 0.01,
    beta1         = b1,
    anchor_power  = 2,
    k_critic      = 1,
    k_generator   = 1,
    warmup_steps  = 30_000/256 * 30,
    warmup_k_critic = 5,
    num_epochs    = 200,
    seed          = my_seed,
    run_name      = 'A8/g1e5_r10_power2_ema99',
    device        = device,
    reset_every   = 10,
    ema_decay     = .99,
)

## Model B: Anchored Optimistic Adam
Shared: γ=3e-6, kc=2, lr=1e-4, reset=10, 200 epochs.

#### **B1 / g3e6_sigSweep**
σ ∈ {1e-3, 5e-3, 1e-2, 5e-2, 1e-1}. fG=128, fD=128. 30k MNIST.

In [ ]:
# # ─── sweep ───────────────────────────────────────────────────────────────
# sigma_sweep_values = [0.001, 0.005, 0.01, 0.05, 0.1]

# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# for sigma_val in sigma_sweep_values:
#     tag = f"sigma{sigma_val}".replace('.', 'p')
#     print(f"\n{'='*72}")
#     print(f"  E39 / σ-sweep on γ=3e-6 + aoadam base — σ = {sigma_val}")
#     print(f"{'='*72}\n")

#     train_loader, test_loader = get_mnist_loaders(
#         batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#         img_size=IMG_SIZE, label=None,
#     )

#     G = MNISTGenerator(channels_img=1, features_g=128).to(device)
#     D = MNISTCritic(channels_img=1, features_d=128, img_size=IMG_SIZE).to(device)

#     G, D = train_mnist(
#         G, D, anchored_optimistic_adam_update, train_loader,
#         resume_G_path=None,
#         resume_D_path=None,
#         initial_noise_type = "low_freq",
#         gamma         = 3e-6,
#         eta           = 1e-4,
#         sigma         = sigma_val,
#         beta1         = b1,
#         anchor_power  = 1,
#         k_critic      = 2,
#         k_generator   = 1,
#         warmup_steps  = 30_000/256 * 30,
#         warmup_k_critic = 5,
#         num_epochs    = 200,
#         seed          = my_seed,
#         run_name      = f'B1/sigSweep_{tag}',
#         device        = device,
#         reset_every   = 10,
#         ema_decay     = None,
#     )


#### **B2 / g3e6_fG128_fD256**
σ=1e-3. fG=128, fD=256. 30k MNIST.

In [ ]:


# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=128).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = f'B2/g3e6_fG128_fD256',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )

#### **B3 / g3e6_fG256_fD256**
σ=1e-3. fG=256, fD=256. 30k MNIST.

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = f'B3/g3e6_fG256_fD256',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )

#### Best model: **B4 / g3e6_fG256_fD256_60k**
σ=1e-3. fG=256, fD=256. **60k MNIST**.
warmup=3,515 steps (~15 ep at 60k vs ~30 ep at 30k).

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=60_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = f'B4/g3e6_fG256_fD256_60k',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
# )

## B5

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = f'B5/g3e6_fG256_fD256_ema999',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = 0.999,
# )


In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 256
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     initial_noise_type = "low_freq",
#     gamma         = 5e-6,        # ← changed
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'B6/g5e6_fG256_fD256_ema99',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = 0.99,
# )


In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 128  # Reduced batch size to mitigate OutOfMemoryError
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 3e-7,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/BATCH_SIZE * 30, # Adjusted warmup steps due to batch size change
#     num_epochs    = 200,
#     run_name      = 'B7g3e6_eta3e-7_fG256_fD256_ema99_noreset',
#     device        = device,
#     reset_every   = 0,           # ← TURN OFF RESETS. Let anchoring/optimism do the work.
#     ema_decay     = 0.99,        # ← Match horizon to the slower updates
# )

In [ ]:
# IMG_SIZE   = 32
# BATCH_SIZE = 128  # Reduced batch size to mitigate OutOfMemoryError
# b1 = 0.5

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = MNISTGenerator(channels_img=1, features_g=256).to(device)
# D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist(
#     G, D, anchored_optimistic_adam_update, train_loader,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 3e-7,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/BATCH_SIZE * 30,
#     num_epochs    = 200,
#     run_name      = 'B7/b/g3e6_eta3e-7_fG256_fD256_ema99_reset',
#     device        = device,
#     reset_every   = 10,           # ← TURN OFF RESETS. Let anchoring/optimism do the work.
#     ema_decay     = 0.999,        # ← Match horizon to the slower updates
# )

### B9 (B6 + NO EMA)

In [2]:
IMG_SIZE   = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None, device=device,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None,
    resume_D_path=None,
    initial_noise_type = "low_freq",
    gamma         = 5e-6,
    eta           = 1e-4,
    sigma         = 0.01,
    beta1         = b1,
    anchor_power  = 1,
    k_critic      = 1,
    k_generator   = 1,
    warmup_steps  = 30_000/256 * 30,
    warmup_k_critic = 5,
    num_epochs    = 200,
    seed          = my_seed,
    run_name      = 'B9/g5e6_fG256_fD256_ema99',
    device        = device,
    reset_every   = 10,
    ema_decay     = .99,
)


G params: 49,813,504
D params: 11,016,321
Steps/epoch: 2  |  Total steps: 6
EMA enabled: decay=0.99  (effective window ≈ 99 G-steps)


Epochs: 100%|██████████| 3/3 [01:47<00:00, 35.71s/it]


### B10: B9+EMA999

In [ ]:
IMG_SIZE   = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

def _count(m): return sum(p.numel() for p in m.parameters())
print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    initial_noise_type = "low_freq",
    gamma         = 5e-6,        # ← changed
    eta           = 1e-4,
    sigma         = .001,
    beta1         = b1,
    anchor_power  = 1,
    k_critic      = 2,
    k_generator   = 1,
    warmup_steps  = 30_000/256 * 30,
    warmup_k_critic = 5,
    num_epochs    = 200,
    seed          = my_seed,
    run_name      = 'B10/g5e6_fG256_fD256_noema',
    device        = device,
    reset_every   = 10,
    ema_decay     = .999,
)


In [ ]:
IMG_SIZE   = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=60_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

def _count(m): return sum(p.numel() for p in m.parameters())
print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    initial_noise_type = "low_freq",
    gamma         = 5e-6,        # ← changed
    eta           = 1e-4,
    sigma         = .001,
    beta1         = b1,
    anchor_power  = 1,
    k_critic      = 2,
    k_generator   = 1,
    warmup_steps  = 60_000/256 * 30,
    warmup_k_critic = 5,
    num_epochs    = 200,
    seed          = my_seed,
    run_name      = 'B10/g5e6_fG256_fD256_ema99_60k',
    device        = device,
    reset_every   = 10,
    ema_decay     = 0.99,
)


## Model C: Conditional OT-GAN
Class-conditional OT-GAN. The generator takes a digit label as input and the discriminator uses projection conditioning

1. Channel-concat conditioning (used in  G)

2. Projection discriminator (used in  D)

### 3.1 `CondGenerator` and `ProjCritic`
The conditional generator stacks a per-class spatial map as an extra input channel so label info flows through every encoder layer. The projection critic adds a class-dependent inner-product term to the standard score.

See `CondGenerator` / `ProjCritic` in `src/models.py`.

### 3.2 Conditional visualization helper
Draws a grid of samples with one digit class per row, used to monitor whether all classes are being learned.

See `plot_conditional_grid` in `src/plotting.py`.

### 3.3 `train_mnist_cond()` loop
Training loop adapted for conditional models: passes labels through G and D, tracks per-class diversity, and stops early if a class collapses.

See `train_mnist_cond` in `src/training.py`.

### C1 / cond_30k_fG256_fD256
Conditional using B3 hyperparameters (fG=256, fD=256, σ=1e-3, γ=3e-6, 30k MNIST, 200 epochs).

In [ ]:
# IMG_SIZE    = 32
# BATCH_SIZE  = 256
# b1          = 0.5
# NUM_CLASSES = 10

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = CondGenerator(channels_img=1, features_g=256,
#                   num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(device)
# D = ProjCritic(channels_img=1, features_d=256,
#                img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist_cond(
#     G, D, anchored_optimistic_adam_update, train_loader, test_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'C1/cond_30k_fG256_fD256_chanCat_realY_1',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
#     num_classes   = NUM_CLASSES,
# )


### C2 / C2_cond_30k_fG256_fD256
Conditional using B3 hyperparameters (fG=256, fD=256, σ=1e-3, γ=3e-6, 60k MNIST, 200 epochs).

In [ ]:
# IMG_SIZE    = 32
# BATCH_SIZE  = 256
# b1          = 0.5
# NUM_CLASSES = 10

# train_loader, test_loader = get_mnist_loaders(
#     batch_size=BATCH_SIZE, device=device, subset_size=60_000, seed=my_seed,
#     img_size=IMG_SIZE, label=None,
# )

# G = CondGenerator(channels_img=1, features_g=256,
#                   num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(device)
# D = ProjCritic(channels_img=1, features_d=256,
#                img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(device)

# def _count(m): return sum(p.numel() for p in m.parameters())
# print(f'G params: {_count(G):,}')
# print(f'D params: {_count(D):,}')

# G, D = train_mnist_cond(
#     G, D, anchored_optimistic_adam_update, train_loader, test_loader,
#     resume_G_path=None,
#     resume_D_path=None,
#     initial_noise_type = "low_freq",
#     gamma         = 3e-6,
#     eta           = 1e-4,
#     sigma         = .001,
#     beta1         = b1,
#     anchor_power  = 1,
#     k_critic      = 2,
#     k_generator   = 1,
#     warmup_steps  = 30_000/256 * 30,         # Make sure to change this when rerunning
#     warmup_k_critic = 5,
#     num_epochs    = 200,
#     seed          = my_seed,
#     run_name      = 'C2/C2_cond_60k_fG256_fD256_chanCat_realY',
#     device        = device,
#     reset_every   = 10,
#     ema_decay     = None,
#     num_classes   = NUM_CLASSES,
# )


## Model CW: Conditional WGAN-GP (cWGAN-GP baseline)

### 4.1 `cond_gradient_penalty` and `train_cond_wgan_gp`

The gradient penalty needs one conceptual change for the conditional case. The picture to have in mind: the critic now scores *(image, label)* pairs, and the Lipschitz constraint is enforced **along the image direction only, at a fixed label**. We draw a random point x̂ on the straight line between a real and a fake image (both belonging to the same label y) and push ‖∇ₓD(x̂, y)‖ toward 1 there. Labels are discrete, so they are never interpolated — each class gets its own Lipschitz-constrained critic slice.

See `cond_gradient_penalty` / `train_mnist_cond_wgan_gp` in `src/training.py`.

### CW1 / cwgan_gp_30k_fG256_fD256

Matches **C1** (30k subset, fG=256, fD=256, batch 256, low-freq noise) and **W1** (all WGAN-GP hyperparameters). 200 epochs ≈ 23.6k generator steps with 5 critic steps each.


In [2]:
# ── Run: CW1 — conditional WGAN-GP baseline matching C1 + W1 (30k MNIST) ─────
IMG_SIZE    = 32
BATCH_SIZE  = 256
NUM_CLASSES = 10

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_0, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = CondGenerator(channels_img=1, features_g=128,
                  num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(device)
D = ProjCritic(channels_img=1, features_d=128,
               img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(device)

def _count(m): return sum(p.numel() for p in m.parameters())
print(f'G params: {_count(G):,}')
print(f'D params: {_count(D):,}')

G, D = train_mnist_cond_wgan_gp(
    G, D, train_loader,
    num_epochs  = 3,
    n_critic    = 2,
    lambda_gp   = 10.0,
    lr          = 1e-4,
    beta1       = 0.0,
    beta2       = 0.9,
    img_size    = IMG_SIZE,
    seed        = my_seed,
    num_classes = NUM_CLASSES,
    run_name    = "CW1/cwgan_gp_30k_fG256_fD256_lambda10_seed42",
    device      = device,
)


G params: 49,825,792
D params: 11,026,561
Steps/epoch: 2  |  Total steps: 6
cWGAN-GP: n_critic=2, lambda_gp=10.0, Adam(lr=0.0001, betas=(0.0, 0.9))


Epochs: 100%|██████████| 3/3 [03:11<00:00, 63.99s/it]


## Z. New experiments (2026): optimizer ladder + reset-at-recipe (30k, B9 base, gamma=5e-6)

Each varies exactly ONE knob. The already-trained AOAdam run at this recipe IS B9 - it is the ladder's 4th rung and the reset arm's reset_every=10 point; do NOT re-train it.

### Ladder rung 1/4 - Adam

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='LADDER/01_adam_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### Ladder rung 2/4 - Optimistic Adam

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='LADDER/02_optadam_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### Ladder rung 3/4 - Anchored Adam (reset_every=10)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='LADDER/03_anchadam_r10_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### Reset ablation - no reset (reset_every=99999 ⇒ fixed-anchor Halpern; anchor stays at init, momentum never zeroed)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='RESET/noreset_aoadam_g5e6_256_30k', device=device,
    reset_every=99999, ema_decay=None,
)


### Reset ablation - reset_every=50

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='RESET/r50_aoadam_g5e6_256_30k', device=device,
    reset_every=50, ema_decay=None,
)


In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

# L2 + EMA — byte-for-byte the optimistic-Adam rung, with EMA(0.99) added.
# Only two lines differ from L2: ema_decay and run_name. Clean one-knob comparison.
G, D = train_mnist(
    G, D, optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='LADDER/02e_optadam_ema99_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=0.99,        # <-- only changes vs L2 (was ema_decay=None)
)

## Z2. TTUR direction test (clean) - critic held at stable γ=5e-6, generator slowed (η sweep)

Tests the *timescale direction* (TTUR puts the **critic** on the faster timescale) WITHOUT crossing the γ-divergence cliff. The critic stays at its known-stable γ=5e-6 and the **generator is slowed** (η ↓) so the critic becomes the faster learner, while the recipe stays inside the bounded-gradient regime. Compare against B9 (η=1e-4). Do **not** test this by raising the critic γ to 1e-4: that conflates the *ratio* with the documented divergence at high γ (untamed |CL|max 10⁵–10⁸; even γ=1e-5 drives critic-grad to 10⁵–10⁶), so a failure there would be uninterpretable. Critic effective rate ≈ γ·k_critic = 5e-6·2 = 1e-5 per gen-step, so η<1e-5 makes the critic faster.

### TTUR-dir 1/2 - generator slowed to η=1e-6 (critic ≈ 10× faster)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=3e-4, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='TTUR/eta1e6_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-6, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='TTUR/eta1e6_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### TTUR-dir 2/2 - generator slowed to η=3e-6 (critic ≈ 3× faster)

> **Note:** the original notebook has a typo here (`asigma=` instead of `sigma=`), which would raise a `TypeError` if run as written -- kept unchanged rather than silently corrected. Let me know if you'd like it fixed.

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=3e-6, asigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='TTUR/eta3e6_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### β₁=0.0 - strip first-moment momentum, keep optimistic look-ahead (raw / no-EMA)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.0

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=2, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='BETA1/b0_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


## Z4. Inner-loop (k_critic) ablation - WGAN/Neural-OT use 5–10; does B9 need it?

Varies only the post-warmup critic:generator step ratio (warmup_k_critic=5 held fixed). B9 uses **k_critic=2**. WGAN (n_critic=5) and Neural-OT solvers train the critic near-optimal each step (5–10 inner steps); if k_critic=2 matches or beats k_critic=5 here, the expensive inner loop is unnecessary in this setting. **Caveat:** this 1-D sweep (AOAdam fixed) supports the *descriptive* claim "AOAdam works at k_critic=2", NOT the causal claim "optimism replaces the inner loop" — that needs k_critic crossed with the optimizer (e.g. add a k=5 point to the Anchored-Adam ladder rung). Report **measured wall-clock**, not an inferred 5× speedup.

### k_critic = 1 - single critic step per generator step (cheapest)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=1, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='KCRITIC/kc1_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)


### k_critic = 5 - WGAN-standard inner loop (most expensive)

In [ ]:
IMG_SIZE = 32
BATCH_SIZE = 256
b1 = 0.5

train_loader, test_loader = get_mnist_loaders(
    batch_size=BATCH_SIZE, device=device, subset_size=30_000, seed=my_seed,
    img_size=IMG_SIZE, label=None,
)

G = MNISTGenerator(channels_img=1, features_g=256).to(device)
D = MNISTCritic(channels_img=1, features_d=256, img_size=IMG_SIZE).to(device)

G, D = train_mnist(
    G, D, anchored_optimistic_adam_update, train_loader,
    resume_G_path=None, resume_D_path=None, initial_noise_type="low_freq",
    gamma=5e-6, eta=1e-4, sigma=0.001, beta1=b1, anchor_power=1,
    k_critic=5, k_generator=1, warmup_steps=30_000/256*30, warmup_k_critic=5,
    num_epochs=200, seed=my_seed, run_name='KCRITIC/kc5_g5e6_256_30k', device=device,
    reset_every=10, ema_decay=None,
)
